# 07 · Tier 4: learning using privileged information (graph → tabular student)

One unit per dataset × graph builder × seed. The graph is treated as *privileged information*: available when
training, absent when scoring. Following Lopez-Paz et al. (2016):

1. **Teacher.** A GNN trained on the *clean* graph (train + val rows only, no test node anywhere) produces a logit
   for every training row.
2. **Student.** XGBoost on tabular features only, trained on the mixed target
   \((1-\lambda)\,y + \lambda\,\sigma(z_{teacher}/T)\). Because `binary:logistic` cross-entropy is linear in the target,
   this equals the two-term distillation loss. \(\lambda = 0\) is the plain Tier 0 XGBoost.
3. **Selection** of \((\lambda, T)\) on validation AUC with real labels. **Evaluation** on test rows, tabular only.

Controls that run inside every unit:

- **self-distillation**: the teacher is a tabular XGBoost with no graph. If it gives the same lift, the graph did nothing.
- the **random_control** graph builder unit (a separate unit of this notebook) is the second control: a GNN teacher on a
  degree-preserving rewired graph. The report compares the two.

For kNN and random-control builders the sweep also runs over k ∈ {5, 10, 20}. Every grid point is checkpointed in
`grid.json`; a killed job continues where it stopped.

**How this notebook runs.** Never by hand in an editor. `scripts/run_nb.sh 07_lupi_distill dataset=... seed=...` executes it
with papermill on the GPU node, writes the executed copy to `runs/07_lupi_distill/<dataset>/<graph_builder>/<seed>/executed.ipynb`,
and the last cell writes a `DONE` marker. If the process dies, the wrapper writes `FAILED` and the next `run_stage.sh`
picks the unit up again; checkpoints inside the run directory mean little work is repeated.

In [ ]:
# Parameters (papermill overrides these; keep names and types)
dataset = "taiwan"
seed = 0
graph_builder = "knn"
run_dir = ""       # empty -> runs/<stage>/<dataset>/<graph_builder>/<seed>
force = 0          # 1 -> redo work even if checkpoints exist
quick = 0          # 1 -> subsampled data, few epochs; smoke test only

In [ ]:
import json
import shutil
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import ckpt, config, paths, viz
from src.eval import results
from IPython.display import display

STAGE = "07_lupi_distill"
warnings.filterwarnings("ignore")
seed = int(seed)
force, quick = bool(int(force)), bool(int(quick))
run_dir = Path(run_dir) if run_dir else paths.run_dir(STAGE, dataset, graph_builder, seed)
run_dir.mkdir(parents=True, exist_ok=True)
t0 = time.time()
print(f"stage={STAGE} dataset={dataset} graph_builder={graph_builder} seed={seed} quick={quick} force={force}")
print("run_dir:", run_dir)
pd.Series(paths.describe())

In [ ]:
from src.data import splits
from src.data.base import Bundle

cfg = config.load_config(dataset)
models_cfg = config.quick_override(cfg, quick)
bundle = Bundle.load(paths.processed_dir(dataset, quick))
split = splits.load_split(dataset, quick)
X, y = bundle.X, bundle.y.to_numpy()
tr, va, te = split["train"], split["val"], split["test"]
allowed = np.sort(np.concatenate([tr, va]))          # rows that may take part in graph construction
X_tr, X_va, X_te = X.iloc[tr], X.iloc[va], X.iloc[te]
y_tr, y_va, y_te = y[tr], y[va], y[te]
gdir = paths.graphs_dir(dataset, quick)
n_boot = 200 if quick else cfg["bootstrap"]["n_boot"]
print(bundle.summary())
print({p: len(split[p]) for p in splits.PARTS}, "| test positive rate:", round(float(np.nanmean(y_te)), 4))

## Teachers on clean graphs

In [ ]:
from src import device
from src.eval import metrics
from src.graphs import assert_no_test_nodes, load_edges
from src.graphs import relational as REL
from src.models import distill as DS
from src.models import gnn_archs as ARCH
from src.models import gnn_train as GT
from src.models import xgb as XGB
from src.models.prep import TabularPrep

dev = device.torch_device()
gcfg = dict(models_cfg["gnn"])
lupi = cfg["lupi"]
xgb_params = dict(models_cfg["xgb"])
prep = TabularPrep().fit(X_tr)
Z = prep.transform(X)
lams, Ts = list(lupi["lambda"]), list(lupi["temperature"])
ks = list(lupi["k"]) if graph_builder in {"knn", "random_control"} else [None]
if quick:
    lams, Ts, ks = [0.0, 0.5, 1.0], [1.0, 2.0], ks[:1]
print("device", dev, "| lambda", lams, "| T", Ts, "| k", ks)


def train_teacher(tag, edges=None, hetero=None):
    """GNN teacher on a clean graph. Returns (logits_train, logits_val, val_auc). Cached per tag."""
    cache = run_dir / "teachers" / f"{tag}.npz"
    if cache.exists() and not force:
        z = np.load(cache)
        return z["train"], z["val"], float(z["val_auc"])
    ck = ckpt.TrainingCheckpoint(run_dir, tag=f"teacher_{tag}", every=gcfg.get("ckpt_every", 1))
    if force:
        shutil.rmtree(ck.dir, ignore_errors=True)
        ck = ckpt.TrainingCheckpoint(run_dir, tag=f"teacher_{tag}", every=gcfg.get("ckpt_every", 1))
    if hetero is not None:
        data, in_dims, nrel = GT.make_hetero_data(hetero, y)
        model = ARCH.build_model("rgcn", 0, gcfg, in_dims=in_dims, num_relations=nrel)
    else:
        data = GT.make_data(Z, y, edges)
        model = ARCH.build_model(lupi.get("teacher", "sage"), Z.shape[1], gcfg)
    GT.train_gnn(model, data, tr, va, gcfg, ck, seed=seed, device=dev, pos_weight=gcfg.get("pos_weight"), log=lambda *a: None)
    lt, _ = GT.predict_gnn(model, data, tr, gcfg, dev, want_emb=False)
    lv, _ = GT.predict_gnn(model, data, va, gcfg, dev, want_emb=False)
    v_auc = metrics.auc(y_va, GT.sigmoid(lv))
    cache.parent.mkdir(exist_ok=True)
    np.savez(cache, train=lt, val=lv, val_auc=v_auc)
    return lt, lv, v_auc


teachers = {}   # name -> (logits_train, logits_val)
teacher_info = {}
t = time.time()
for k in ks:
    name = graph_builder + (f"_k{k}" if k else "")
    if graph_builder == "relational":
        hetero = REL.load_hetero(gdir / "hetero_relational_clean.npz")
        hetero["x_customer"] = Z
        for _, rel, _ in REL.RELATIONS:
            assert not np.isin(hetero[f"ei_customer__{rel}"][0], te).any(), "LEAKAGE in clean hetero graph"
        lt, lv, v_auc = train_teacher(name, hetero=hetero)
    else:
        e = load_edges(dataset, graph_builder, cfg, quick, k=k, variant="clean")
        assert_no_test_nodes(e, te, name)
        lt, lv, v_auc = train_teacher(name, edges=e)
    teachers[name] = (lt, lv)
    teacher_info[name] = {"kind": "gnn", "val_auc": v_auc, **DS.teacher_agreement(lt, y_tr)}
    print(f"teacher {name}: val AUC {v_auc:.4f}")

# self-distillation control: tabular XGBoost teacher (in-sample train logits, as in standard self-distillation)
cache = run_dir / "teachers" / "self_xgb.npz"
if cache.exists() and not force:
    z = np.load(cache); lt, lv = z["train"], z["val"]
else:
    b0 = XGB.fit_xgb(X_tr, y_tr, X_va, y_va, xgb_params, seed=seed)
    lt, lv = DS.logits_from_probs(XGB.predict_xgb(b0, X_tr)), DS.logits_from_probs(XGB.predict_xgb(b0, X_va))
    cache.parent.mkdir(exist_ok=True); np.savez(cache, train=lt, val=lv)
teachers["self_xgb"] = (lt, lv)
teacher_info["self_xgb"] = {"kind": "tabular", "val_auc": metrics.auc(y_va, GT.sigmoid(lv)), **DS.teacher_agreement(lt, y_tr)}
print(f"teachers ready in {time.time() - t:.0f}s")
pd.DataFrame(teacher_info).T

## Student grid

One XGBoost per (teacher, λ, T). λ = 0 is fit once and shared (it is the plain baseline). Each finished point is written to `grid.json` immediately.

In [ ]:
grid = ckpt.GridCheckpoint(run_dir / "grid.json")
(run_dir / "students").mkdir(exist_ok=True)
plan = [("none", 0.0, 1.0)] + [(tn, lam, T) for tn in teachers for lam in lams if lam > 0 for T in Ts]
print(f"{len(plan)} grid points, {len(grid)} already done")


def student_file(key):
    return run_dir / "students" / (key.replace("|", "__").replace("=", "-") + ".npz")


t = time.time()
for tname, lam, T in plan:
    key = grid.key(teacher=tname, lam=lam, T=T)
    if grid.done(key) and student_file(key).exists() and not force:
        continue
    lt = teachers[tname][0] if tname != "none" else np.zeros(len(tr), np.float32)
    t1 = time.time()
    b = DS.train_student(X_tr, y_tr, lt, lam, T, X_va, y_va, xgb_params, seed=seed)
    pv, pt = XGB.predict_xgb(b, X_va), XGB.predict_xgb(b, X_te)
    np.savez_compressed(student_file(key), val=pv, test=pt)
    grid.record(key, {"teacher": tname, "lam": lam, "T": T, "val_auc": metrics.auc(y_va, pv),
                      "test": metrics.all_metrics(y_te, pt), "best_iteration": XGB.summary(b)["best_iteration"],
                      "seconds": round(time.time() - t1, 1)})
    print(f"  {key:40s} val AUC {metrics.auc(y_va, pv):.4f}  test AUC {metrics.auc(y_te, pt):.4f}")
print(f"grid complete in {time.time() - t:.0f}s")
res = pd.DataFrame([{**{k: v for k, v in r.items() if k != "test"}, **{f"test_{k}": v for k, v in r["test"].items()}} for r in grid.results().values()])
res.sort_values(["teacher", "lam", "T"]).reset_index(drop=True)

## Select on validation, report on test

For each teacher the (λ, T) with the best validation AUC is the *student*. Its test predictions are compared with the plain XGBoost (λ = 0) on the same rows.

In [ ]:
plain_key = grid.key(teacher="none", lam=0.0, T=1.0)
plain = np.load(student_file(plain_key))
selected = {}
for tname in teachers:
    sub = res[(res.teacher == tname) & (res.lam > 0)]
    if sub.empty:
        continue
    best = sub.sort_values("val_auc", ascending=False).iloc[0]
    selected[tname] = {"lam": float(best.lam), "T": float(best["T"]), "val_auc": float(best.val_auc), "test_auc": float(best.test_auc)}
sel_preds = {f"student_{tn}": np.load(student_file(grid.key(teacher=tn, lam=s["lam"], T=s["T"])))["test"] for tn, s in selected.items()}

from src.eval import bootstrap

compare = bootstrap.compare_many(y_te, plain["test"], sel_preds, metrics=("auc", "pr_auc", "capture_10"), n_boot=n_boot, seed=seed)
for r in compare:
    r["baseline"] = "xgb_plain"
cmp_df = pd.DataFrame(compare)
fig = viz.plot_intervals(cmp_df[cmp_df.metric == "auc"], title=f"Student test AUC minus plain XGBoost ({dataset}/{graph_builder}, seed {seed})")
viz.savefig(fig, run_dir, "student_gain")
plt.show()
display(pd.DataFrame(selected).T.round(4))
cmp_df[["model", "metric", "base", "new", "diff", "ci_low", "ci_high", "significant"]].round(4)

## The whole grid

Test AUC over (λ, T) per teacher. A flat surface means the student ignores the teacher; a ridge at λ > 0 means it learns from it.

In [ ]:
teachers_plot = [tn for tn in teachers if not res[res.teacher == tn].empty]
fig, axes = plt.subplots(1, len(teachers_plot), figsize=(4.2 * len(teachers_plot), 3.4), squeeze=False)
plain_auc = float(res.loc[res.teacher == "none", "test_auc"].iloc[0])
for ax, tn in zip(axes[0], teachers_plot):
    piv = res[res.teacher == tn].pivot_table(index="T", columns="lam", values="test_auc")
    piv.insert(0, 0.0, plain_auc)
    im = ax.imshow(piv.to_numpy(), aspect="auto", cmap="viridis")
    ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns); ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index)
    ax.set(title=f"{tn}\ntest AUC", xlabel="lambda", ylabel="T")
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            ax.text(j, i, f"{piv.iloc[i, j]:.3f}", ha="center", va="center", color="w", fontsize=7)
fig.tight_layout()
viz.savefig(fig, run_dir, "grid_heatmap")
plt.show()

In [ ]:
results.save_predictions(run_dir, "xgb_plain", {"val": (va, y_va, plain["val"]), "test": (te, y_te, plain["test"])})
rows = {"xgb_plain": results.metrics_block(y_va, plain["val"], y_te, plain["test"])}
for tn, s in selected.items():
    z = np.load(student_file(grid.key(teacher=tn, lam=s["lam"], T=s["T"])))
    results.save_predictions(run_dir, f"student_{tn}", {"val": (va, y_va, z["val"]), "test": (te, y_te, z["test"])})
    rows[f"student_{tn}"] = results.metrics_block(y_va, z["val"], y_te, z["test"])
results.save_metrics(run_dir, {"models": rows, "compare": compare,
                               "lupi": {"selected": selected, "teachers": teacher_info, "grid": res.to_dict(orient="records"),
                                        "lambda": lams, "temperature": Ts, "k": ks},
                               "params": {"graph_builder": graph_builder, "seed": seed, "quick": quick}})
summary = {"plain_test_auc": round(rows["xgb_plain"]["test"]["auc"], 4),
           "student_test_auc": {tn: round(s["test_auc"], 4) for tn, s in selected.items()}, "elapsed_s": round(time.time() - t0, 1)}
ckpt.mark_done(run_dir, summary)
summary